In [23]:
import numpy as np
import pandas as pd
from pathlib import Path

## 1. 데이터 로드


In [24]:
ROOT = Path('../../')
RAW_DIR = ROOT / 'data' / 'raw'

df_f = pd.read_csv(RAW_DIR / 'sports_gb_F.csv')
df_l = pd.read_csv(RAW_DIR / 'sports_gb_L.csv')
df_t = pd.read_csv(RAW_DIR / 'sports_gb_total.csv')

## 2. 날짜 파싱


In [25]:
for col in ['DateReg', 'Date1Dep', 'Date1Spo']:
    df_t[col] = pd.to_datetime(df_t[col])

df_f['DateBet'] = pd.to_datetime(df_f['DateBet'])
df_l['DateBet'] = pd.to_datetime(df_l['DateBet'])

## 3. 기본 feature 생성

`sports_gb_total.csv`에서 직접적으로 가져올 수 있는 feature rename


In [26]:
df = df_t[[
    'UserID', 'CountryID', 'Gender', 'BirthYear', 'DateReg', 'Date1Dep', 'Date1Spo',
    'StakeF', 'StakeL', 'StakeA',
    'WinF',   'WinL',   'WinA',
    'BetsF',  'BetsL',  'BetsA',
    'DaysF',  'DaysL',  'DaysA',
]].copy()

df = df.rename(columns={
    'UserID'    : 'user_id',
    'CountryID' : 'country_id',
    'Gender'    : 'gender',
    'BirthYear' : 'birth_year',
    'DateReg'   : 'reg_date',
    'Date1Dep'  : 'first_deposit',
    'Date1Spo'  : 'first_bet',
    'StakeF'    : 'fixed_bet_amount',
    'StakeL'    : 'live_bet_amount',
    'StakeA'    : 'total_bet_amount',
    'WinF'      : 'fixed_win_amount',
    'WinL'      : 'live_win_amount',
    'WinA'      : 'total_win_amount',
    'BetsF'     : 'fixed_bet_cnt',
    'BetsL'     : 'live_bet_cnt',
    'BetsA'     : 'total_bet_cnt',
    'DaysF'     : 'fixed_active_days',
    'DaysL'     : 'live_active_days',
    'DaysA'     : 'total_active_days',
})

df.head()

,user_id,country_id,gender,birth_year,reg_date,first_deposit,first_bet,fixed_bet_amount,live_bet_amount,total_bet_amount,fixed_win_amount,live_win_amount,total_win_amount,fixed_bet_cnt,live_bet_cnt,total_bet_cnt,fixed_active_days,live_active_days,total_active_days
0,1324354,276,1,1963.0,2005-02-01,2005-02-24,2005-02-24,15750.3800,2146.4700,17896.8500,15010.9000,1809.9500,16820.8500,727,71,798,231,33,233
1,1324355,300,1,1983.0,2005-02-01,2005-02-01,2005-02-01,639.2998,24.7000,663.9998,569.3700,11.2000,580.5700,286,21,307,99,7,101
2,1324356,276,1,1977.0,2005-02-01,2005-02-01,2005-02-02,898.8100,701.8200,1600.6300,336.3600,649.2700,985.6300,116,126,242,48,27,54
3,1324358,752,1,1981.0,2005-02-01,2005-02-01,2005-02-01,247.6970,88.5927,336.2897,153.8755,55.9819,209.8574,7,4,11,5,1,5
4,1324360,792,1,1978.0,2005-02-01,2005-02-02,2005-02-02,685.9424,6.6641,692.6065,623.8984,3.0528,626.9512,386,8,394,58,4,60


## 4. age_group 계산

기준년도: 2006년
10대:0 / 20대:1 / 30대:2 / 40대:3 / 50대:4 / 60대:5 / 70대:6 / 80대:7 / 90대이상:8


In [27]:
# 한국 나이 기준
REFERENCE_YEAR = 2006 + 1

df['age'] = REFERENCE_YEAR - df['birth_year']

bins = [-np.inf, 19, 29, 39, 49, 59, 69, 79, 89, np.inf]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False).astype('Int64').fillna(-1)

# 필요없어진 컬럼 제거
df = df.drop(columns=['birth_year', 'age'])

In [28]:
df['age_group'].value_counts().sort_index()

age_group
-1        2
0         4
1     18186
2     16497
3      7681
4      2998
5       772
6       159
7        35
8         5
Name: count, dtype: Int64

## 5. hit_days 계산

`hit_days = (Win > 0) / (Bets > 0)`

- **Fixed/Live**: 각 유형별 독립 계산
- **Total**: F + L 일별 합산 후 판정


In [29]:
# fixed_hit_days
_f = df_f.rename(columns={'UserID': 'user_id'})

fixed_hit_days = (
  _f[_f['BetsF'] > 0]
  .groupby('user_id')['WinF']
  .apply(lambda s: (s > 0).sum())
  .rename('fixed_hit_days')
  .reset_index()
)

fixed_hit_days

,user_id,fixed_hit_days
0,1324354,93
1,1324355,15
2,1324356,12
3,1324358,1
4,1324360,14
...,...,...
45604,1405183,1
45605,1405184,11
45606,1405185,1
45607,1405189,32


In [30]:
# live_hit_days
_l = df_l.rename(columns={'UserID': 'user_id'})

live_hit_days = (
  _l[_l['BetsL'] > 0]
  .groupby('user_id')['WinL']
  .apply(lambda s: (s > 0).sum())
  .rename('live_hit_days')
  .reset_index()
)

live_hit_days

,user_id,live_hit_days
0,1324354,22
1,1324355,1
2,1324356,18
3,1324358,1
4,1324360,2
...,...,...
31397,1405182,11
31398,1405183,8
31399,1405184,15
31400,1405185,2


In [31]:
# dataFrame 병합
_fl = (
  pd.merge(
    _f[['user_id', 'DateBet', 'BetsF', 'WinF']],
    _l[['user_id', 'DateBet', 'BetsL', 'WinL']],
    on=['user_id', 'DateBet'],
    how='outer',
  ).fillna(0)
  .assign(
    Bets=lambda df: df['BetsF'] + df['BetsL'],
    Win=lambda df: df['WinF'] + df['WinL'],
  )
)

In [32]:
# total_hit_days
total_hit_days = (
  _fl[_fl['Bets'] > 0]
  .groupby('user_id')['Win']
  .apply(lambda s: (s > 0).sum())
  .rename('total_hit_days')
  .reset_index()
)

fixed_hit_days

,user_id,fixed_hit_days
0,1324354,93
1,1324355,15
2,1324356,12
3,1324358,1
4,1324360,14
...,...,...
45604,1405183,1
45605,1405184,11
45606,1405185,1
45607,1405189,32


## 6. win_rate 계산

`win_rate = (Win >= Stake) / (Bets > 0)`

- **Fixed/Live**: 각 유형별 독립 계산
- **Total**: F + L 일별 합산 후 판정


In [33]:
# fixed_win_rate
fixed_win_rate = (
  _f[_f['BetsF'] > 0]  # fixed 베팅이 있는 경우
  .assign(_win=lambda df: df['WinF'] >= df['StakeF'])
  .groupby('user_id')['_win']
  .mean()  # 승률 계산
  .rename('fixed_win_rate')
  .reset_index()
)

fixed_win_rate

,user_id,fixed_win_rate
0,1324354,0.290043
1,1324355,0.141414
2,1324356,0.125000
3,1324358,0.000000
4,1324360,0.137931
...,...,...
45604,1405183,0.043478
45605,1405184,0.235294
45606,1405185,0.500000
45607,1405189,0.301887


In [34]:
# live_win_rate
live_win_rate = (
  _l[_l['BetsL'] > 0]  # live 베팅이 있는 경우
  .assign(_win=lambda df: df['WinL'] >= df['StakeL'])
  .groupby('user_id')['_win']
  .mean()  # 승률 계산
  .rename('live_win_rate')
  .reset_index()
)

live_win_rate

,user_id,live_win_rate
0,1324354,0.454545
1,1324355,0.142857
2,1324356,0.259259
3,1324358,0.000000
4,1324360,0.250000
...,...,...
31397,1405182,0.400000
31398,1405183,0.307692
31399,1405184,0.391304
31400,1405185,0.200000


In [35]:
# _fl에 Stake가 없으므로 Stake 포함하여 별도 merge
_fl_w = (
  pd.merge(
    _f[['user_id', 'DateBet', 'StakeF', 'WinF', 'BetsF']],
    _l[['user_id', 'DateBet', 'StakeL', 'WinL', 'BetsL']],
    on=['user_id', 'DateBet'],
    how='outer',
  )
  .fillna(0)
  .assign(
    Stake=lambda df: df['StakeF'] + df['StakeL'],
    Win=lambda df: df['WinF'] + df['WinL'],
    Bets=lambda df: df['BetsF'] + df['BetsL'],
  ) 
)

In [36]:
total_win_rate = (
    _fl_w[_fl_w['Bets'] > 0]
    .assign(_win=lambda x: x['Win'] >= x['Stake'])
    .groupby('user_id')['_win']
    .mean()
    .rename('total_win_rate')
    .reset_index()
)

fixed_win_rate

,user_id,fixed_win_rate
0,1324354,0.290043
1,1324355,0.141414
2,1324356,0.125000
3,1324358,0.000000
4,1324360,0.137931
...,...,...
45604,1405183,0.043478
45605,1405184,0.235294
45606,1405185,0.500000
45607,1405189,0.301887


## 7. avg_roi 계산

`avg_roi = mean((Win - Stake) / Stake)` where `Bets > 0`

- 일별 ROI를 먼저 계산한 뒤 유저별 평균 → 베팅 빈도 관계없이 하루하루의 수익률을 동등하게 반영
- **Total**: `_fl_w`(win_rate에서 생성, F + L 일별 합산) 재사용


In [37]:
# fixed_avg_roi
fixed_avg_roi = (
  _f[_f['BetsF'] > 0]
  .assign(_roi=lambda df: (df['WinF'] - df['StakeF']) / df['StakeF'])
  .groupby('user_id')['_roi']
  .mean()
  .rename('fixed_avg_roi')
  .reset_index()
)

fixed_avg_roi

,user_id,fixed_avg_roi
0,1324354,-0.004817
1,1324355,0.171266
2,1324356,0.379324
3,1324358,-0.883832
4,1324360,0.885973
...,...,...
45604,1405183,-0.895155
45605,1405184,-0.455678
45606,1405185,-0.200000
45607,1405189,0.200214


In [38]:
# live_avg_roi
live_avg_roi = (
  _l[_l['BetsL'] > 0]
  .assign(_roi=lambda df: (df['WinL'] - df['StakeL']) / df['StakeL'])
  .groupby('user_id')['_roi']
  .mean()
  .rename('live_avg_roi')
  .reset_index()
)

live_avg_roi

,user_id,live_avg_roi
0,1324354,-0.089255
1,1324355,-0.761194
2,1324356,-0.248959
3,1324358,-0.368098
4,1324360,-0.632391
...,...,...
31397,1405182,-0.169908
31398,1405183,-0.291959
31399,1405184,-0.207444
31400,1405185,-0.588787


In [39]:
# total_avg_roi
total_avg_roi = (
    _fl_w[_fl_w['Bets'] > 0]
    .assign(_roi=lambda x: (x['Win'] - x['Stake']) / x['Stake'])
    .groupby('user_id')['_roi']
    .mean()
    .rename('total_avg_roi')
    .reset_index()
)

total_avg_roi

,user_id,total_avg_roi
0,1324354,0.040771
1,1324355,0.159505
2,1324356,0.001383
3,1324358,-0.791456
4,1324360,1.355822
...,...,...
46334,1405183,-0.621482
46335,1405184,-0.347865
46336,1405185,-0.482120
46337,1405189,0.166778


## 8. churn 계산

기준일: 데이터 마지막 날짜 `2006-08-31`

- `0`: 기준일로부터 13개월(395일) 이내 베팅 활동 있음
- `1`: 없음


In [40]:
REFERENCE_DATE = pd.Timestamp('2006-08-31')

last_bet = (
    pd.concat([
        _f.groupby('user_id')['DateBet'].max().rename('last_f'),
        _l.groupby('user_id')['DateBet'].max().rename('last_l'),
    ], axis=1)
    .max(axis=1)
    .rename('recency')
    .reset_index()
)

last_bet['churn'] = (
    (REFERENCE_DATE - last_bet['recency']).dt.days > 395
).astype(int)

last_bet['churn'].value_counts()

churn
0    33322
1    13017
Name: count, dtype: int64

## 9. 최종 DataFrame 병합


In [41]:
result = df.copy()

for aux in [
    fixed_hit_days,  live_hit_days,  total_hit_days,
    fixed_win_rate,  live_win_rate,  total_win_rate,
    fixed_avg_roi,   live_avg_roi,   total_avg_roi,
]:
    result = result.merge(aux, on='user_id', how='left')

result = result.merge(last_bet[['user_id', 'churn']], on='user_id', how='left')

FINAL_COLS = [
    'user_id', 'country_id', 'gender', 'age_group', 'reg_date', 'first_deposit',
    'first_bet',
    'fixed_bet_amount',  'live_bet_amount',  'total_bet_amount',
    'fixed_win_amount',  'live_win_amount',  'total_win_amount',
    'fixed_bet_cnt',     'live_bet_cnt',     'total_bet_cnt',
    'fixed_active_days', 'live_active_days', 'total_active_days',
    'fixed_hit_days',    'live_hit_days',    'total_hit_days',
    'fixed_win_rate',    'live_win_rate',    'total_win_rate',
    'fixed_avg_roi',     'live_avg_roi',     'total_avg_roi',
    'churn',
]

result = result[FINAL_COLS]

# 소수점 2자리로 반올림
float_cols = result.select_dtypes(include='float').columns.tolist()
result = result.assign(**{col: result[col].round(2) for col in float_cols})

result.shape


(46339, 29)

## 11. 저장


In [42]:
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'ljh_preprocessed.csv'
result.to_csv(OUTPUT_PATH, index=False)